# Temperature Forecasting
**Project 38 of 50 — Time Series Forecasting Portfolio**

## Project Overview
| Attribute | Detail |
|---|---|
| **Kaggle slug** | `sumanthvrao/daily-climate-time-series-data` |
| **Primary file** | `DailyDelhiClimateTrain.csv` |
| **Key column** | `meantemp` |
| **Target** | Daily mean temperature (°C) in Delhi |
| **Frequency** | Daily (D) |
| **Season length** | 365 (annual cycle) |
| **TS library** | Darts (ExponentialSmoothing + trend, Theta) |

## Learning Objectives
1. Model strong annual temperature seasonality with a 365-day period
2. Identify the Delhi climate cycle — hot dry summer, monsoon cooling, cold winter
3. Apply Darts models on a clean environmental daily series
4. Use period=365 decomposition to visualise the annual temperature wave
5. Compare statistical models against FLAML lag-feature regressors
6. Evaluate 28-day ahead temperature forecast accuracy

## Problem Statement
Forecast Delhi daily mean temperature 28 days ahead for agricultural planning, energy demand forecasting, and public health advisories.

## Why This Project Matters
Temperature forecasts drive electricity demand projections (air conditioning loads), irrigation scheduling, and cold-chain logistics. A 28-day outlook at 1°C accuracy exceeds the performance of most simple persistence models.

## Dataset Overview
A clean 4-year (2013–2017) daily climate series for Delhi containing `meantemp`, `humidity`, `wind_speed`, and `meanpressure`.

## Dataset Source & License
- **Kaggle**: https://www.kaggle.com/datasets/sumanthvrao/daily-climate-time-series-data
- **License**: CC0 (Public Domain)

## Environment Setup

In [ ]:
import subprocess, sys
for p in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
          "statsmodels","scikit-learn","lazypredict","flaml",
          "statsforecast","darts"]:
    try:
        __import__(p.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats as scipy_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
pd.set_option("display.max_columns",40)
plt.rcParams["figure.figsize"] = (14,5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration

In [ ]:
PROJECT     = "Temperature Forecasting"
KAGGLE_SLUG = "sumanthvrao/daily-climate-time-series-data"
FREQ        = "D"
SEASON_LEN  = 365
HORIZON     = 28
TEST_SIZE   = HORIZON
VAL_SIZE    = HORIZON
RANDOM_SEED = 42
FLAML_SECS  = 120
DATA_DIR    = pathlib.Path("data/sumanthvrao_daily_climate_time_series_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Horizon={HORIZON} | Season={SEASON_LEN}")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME","KAGGLE_KEY","KAGGLE_API_TOKEN"]:
    if os.environ.get(k):
        kaggle_ok = True; print(f"Credential found: {k}"); break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists():
    kaggle_ok = True; print(f"kaggle.json at {kj}")
if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("https://www.kaggle.com/settings -> Create New Token")
else:
    print("Credentials verified.")

## Dataset Download

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data"))
    print(f"Dataset at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system("kaggle datasets download -d sumanthvrao/daily-climate-time-series-data -p data/sumanthvrao_daily_climate_time_series_data --unzip")
    data_path = DATA_DIR

csvs = sorted(data_path.rglob("*.csv"), key=lambda f: f.stat().st_size, reverse=True)
print(f"CSV files ({len(csvs)}):")
for f in csvs:
    sz = f.stat().st_size / 1e6
    print(f"  {f.name}  ({sz:.2f} MB)")

## Load Data & Build Series

In [ ]:
csv_file = next((f for f in csvs if "train" in f.name.lower()), csvs[0])
print(f"Loading: {csv_file.name}")
df_raw = pd.read_csv(csv_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

In [ ]:
date_col = next(c for c in df_raw.columns if "date" in c.lower())
temp_col = next((c for c in df_raw.columns if "meantemp" in c.lower() or "mean_temp" in c.lower() or "temperature" in c.lower()), None)
if temp_col is None:
    numeric_cols = df_raw.select_dtypes("number").columns.tolist()
    temp_col = numeric_cols[0]
print(f"Date: '{date_col}'  Temp: '{temp_col}'")

df_raw[date_col] = pd.to_datetime(df_raw[date_col], errors="coerce")
df_raw[temp_col] = pd.to_numeric(df_raw[temp_col], errors="coerce")
df_clean = df_raw.dropna(subset=[date_col, temp_col]).copy()

# Train file may be 'DailyDelhiClimateTrain.csv', test 'DailyDelhiClimateTest.csv'
# Use train file; if both available, concatenate
ts_raw = df_clean.rename(columns={date_col:"date", temp_col:"y_val"})[["date","y_val"]]
ts_raw["date"] = pd.to_datetime(ts_raw["date"])
ts_raw = ts_raw.sort_values("date").reset_index(drop=True)
print(f"Series: {len(ts_raw)} days")
print(f"Range: {ts_raw['date'].min().date()} to {ts_raw['date'].max().date()}")
print(ts_raw["y_val"].describe().round(2))

## Data Validation

In [ ]:
print("DATA VALIDATION")
print("="*50)
full_range = pd.date_range(ts_raw["date"].min(), ts_raw["date"].max(), freq=FREQ)
missing = full_range.difference(ts_raw["date"])
print(f"Missing periods : {len(missing)}")
zero_p = (ts_raw["y_val"] == 0).sum()
print(f"Zero-value periods: {zero_p}")
mu, sig = ts_raw["y_val"].mean(), ts_raw["y_val"].std()
outliers = ts_raw[abs(ts_raw["y_val"]-mu) > 3*sig]
print(f"3-sigma outliers: {len(outliers)}")
for _, row in outliers.iterrows():
    print(f"  {row['date'].date()}  value={row['y_val']:.3f}")

## Exploratory Data Analysis

In [ ]:
ts_raw_f = ts_raw.set_index("date").reindex(full_range).rename_axis("date")
ts_raw_f["y_val"] = ts_raw_f["y_val"].interpolate("linear")
ts_raw_f = ts_raw_f.reset_index()
ts_df = ts_raw_f.rename(columns={"date":"ds","y_val":"y"}).copy()
ts_df["y"] = ts_df["y"].astype(float)
print(f"Series: {len(ts_df)} periods")

In [ ]:
# Time series plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"],name="Observed",line=dict(color="#1565C0",width=1.5)))
roll_w = min(28, max(4, len(ts_df)//20))
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"].rolling(roll_w,center=True).mean(),
    name=f"{roll_w}-period MA",line=dict(color="#F44336",width=2.5,dash="dot")))
fig.update_layout(title=f"Time Series — {PROJECT}",xaxis_title="Date",
    yaxis_title="Value",template="plotly_white",legend=dict(orientation="h",y=-0.2))
fig.show()

In [ ]:
# Seasonal decomposition
ts_idx = ts_df.set_index("ds")["y"].asfreq(FREQ)
try:
    decomp = seasonal_decompose(ts_idx.dropna(), model="additive", period=SEASON_LEN)
    fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
    for ax, s, t, c in zip(axes,
        [decomp.observed,decomp.trend,decomp.seasonal,decomp.resid.dropna()],
        ["Observed","Trend",f"Seasonal (period={SEASON_LEN})","Residual"],
        ["#1565C0","#C62828","#2E7D32","#757575"]):
        s.plot(ax=ax,title=t,color=c)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Decomposition error: {e}")

In [ ]:
fig2, axes2 = plt.subplots(1,2,figsize=(14,4))
plot_acf(ts_df["y"].dropna(), lags=min(49,len(ts_df)//2-1), ax=axes2[0], title="ACF")
plot_pacf(ts_df["y"].dropna(), lags=min(49,len(ts_df)//4-1), ax=axes2[1], title="PACF")
plt.tight_layout(); plt.show()
adf = adfuller(ts_df["y"].dropna())
print(f"ADF p={adf[1]:.4f} -> {'stationary' if adf[1]<0.05 else 'non-stationary'}") 

## Target Analysis

In [ ]:
print(ts_df["y"].describe().round(4))
print(f"Skew: {ts_df['y'].skew():.3f}  Kurtosis: {ts_df['y'].kurtosis():.3f}")
fig, axes = plt.subplots(1,2,figsize=(12,4))
ts_df["y"].hist(bins=40,ax=axes[0],edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].set_title("Distribution")
scipy_stats.probplot(ts_df["y"].dropna(),dist="norm",plot=axes[1])
axes[1].set_title("Q-Q Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_train     = ts_df.iloc[:n-28-28].copy()
ts_val       = ts_df.iloc[n-28-28:n-28].copy()
ts_test      = ts_df.iloc[n-28:].copy()
ts_train_val = pd.concat([ts_train,ts_val]).reset_index(drop=True)
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max()   < ts_test["ds"].min()
print(f"Train: {len(ts_train)}  Val: {len(ts_val)}  Test: {len(ts_test)}")
print(f"  Train: {ts_train['ds'].min().date()} – {ts_train['ds'].max().date()}")
print(f"  Test : {ts_test['ds'].min().date()} – {ts_test['ds'].max().date()}")

## Preprocessing — Winsorisation

In [ ]:
lo = ts_train["y"].quantile(0.01)
hi = ts_train["y"].quantile(0.99)
ts_train_w = ts_train.copy()
ts_train_w["y"] = ts_train_w["y"].clip(lo, hi)
ts_tv_w = pd.concat([ts_train_w, ts_val]).reset_index(drop=True)
print(f"Winsorisation: [{lo:.3f}, {hi:.3f}]")

## Feature Engineering

In [ ]:
def make_features(df):
    df = df.copy()
    for lag in [1,2,3,7,14,21,28]: df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in [7,14,28]:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = df["y"].shift(1).rolling(w).std()
    df["dow"]         = df["ds"].dt.dayofweek
    df["month"]       = df["ds"].dt.month
    df["quarter"]     = df["ds"].dt.quarter
    df["weekofyear"]  = df["ds"].dt.isocalendar().week.astype(int)
    df["is_weekend"]  = (df["ds"].dt.dayofweek >= 5).astype(int)
    df["is_monthend"] = df["ds"].dt.is_month_end.astype(int)
    return df

In [ ]:
ts_full = pd.concat([ts_train,ts_val,ts_test]).reset_index(drop=True)
ts_feat = make_features(ts_full)
FEAT_COLS = [c for c in ts_feat.columns if c not in ["ds","y"]]
n_tr,n_va = len(ts_train),len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()
print(f"Features ({len(FEAT_COLS)}): {FEAT_COLS}")

## Baseline Models

In [ ]:
def eval_fc(actual, pred, name):
    a = np.array(actual).flatten()
    p = np.array(pred).flatten()[:len(a)]
    mae = mean_absolute_error(a,p)
    rmse = float(np.sqrt(mean_squared_error(a,p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100) if mask.sum() else float("nan")
    print(f"{name:40s}  MAE={mae:8.3f}  RMSE={rmse:8.3f}  MAPE={mape:6.2f}%")
    return dict(model=name, MAE=mae, RMSE=rmse, MAPE=mape)

In [ ]:
results = []
naive_p = np.full(TEST_SIZE, ts_train_val["y"].iloc[-1])
results.append(eval_fc(ts_test["y"], naive_p, "Naive (last value)"))
sn = ts_train_val["y"].iloc[-SEASON_LEN:].values
sn = np.tile(sn, TEST_SIZE//SEASON_LEN + 1)[:TEST_SIZE]
results.append(eval_fc(ts_test["y"], sn, f"Seasonal Naive (lag-{SEASON_LEN})"))
ma = np.full(TEST_SIZE, ts_train_val["y"].iloc[-7:].mean())
results.append(eval_fc(ts_test["y"], ma, "Moving Average (7-period)"))

## LazyPredict Benchmark

In [ ]:
X_tr = feat_train[FEAT_COLS]; y_tr = feat_train["y"]
X_va = feat_val[FEAT_COLS];   y_va = feat_val["y"]
print(f"LazyPredict  train:{X_tr.shape}  val:{X_va.shape}")
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lz_models, _ = lz.fit(X_tr, X_va, y_tr, y_va)
    print("\nTop 15:")
    print(lz_models.head(15).to_string())
except Exception as e:
    print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_tv = pd.concat([feat_train,feat_val])[FEAT_COLS]
y_tv = pd.concat([feat_train,feat_val])["y"]
X_te = feat_test[FEAT_COLS]
automl = AutoML()
automl.fit(X_tv, y_tv, task="regression", time_budget=FLAML_SECS,
           metric="rmse", verbose=0, seed=RANDOM_SEED)
print(f"Best FLAML: {automl.best_estimator}")
flaml_pred = automl.predict(X_te)
results.append(eval_fc(feat_test["y"], flaml_pred, f"FLAML ({automl.best_estimator})"))

## Darts Section

In [ ]:
try:
    from darts import TimeSeries
    from darts.models import AutoARIMA, ExponentialSmoothing, Theta, NaiveSeasonal
    from darts.metrics import mape as darts_mape, rmse as darts_rmse

    darts_series = TimeSeries.from_dataframe(ts_tv_w, time_col="ds", value_cols="y", freq="D")
    darts_test   = TimeSeries.from_dataframe(ts_test,  time_col="ds", value_cols="y", freq="D")

    darts_models = {
        "Darts-AutoARIMA"        : AutoARIMA(),
        "Darts-ExponentialSmooth": ExponentialSmoothing(trend=True, seasonal_periods=365),
        "Darts-Theta"            : Theta(season_mode="multiplicative"),
        "Darts-NaiveSeasonal"    : NaiveSeasonal(K=365),
    }
    darts_best_pred = None
    for mname, model in darts_models.items():
        try:
            model.fit(darts_series)
            fc = model.predict(n=28)
            fc_vals = fc.values().flatten()
            actual_vals = darts_test.values().flatten()[:len(fc_vals)]
            rmse_v = float(np.sqrt(mean_squared_error(actual_vals, fc_vals)))
            mae_v  = mean_absolute_error(actual_vals, fc_vals)
            mask = actual_vals != 0
            mape_v = float(np.mean(np.abs((actual_vals[mask]-fc_vals[mask])/actual_vals[mask]))*100) if mask.sum() else float("nan")
            print(f"{mname:40s}  MAE={mae_v:8.3f}  RMSE={rmse_v:8.3f}  MAPE={mape_v:6.2f}%")
            results.append(dict(model=mname, MAE=mae_v, RMSE=rmse_v, MAPE=mape_v))
            if darts_best_pred is None:
                darts_best_pred = fc_vals
        except Exception as e:
            print(f"  {mname} failed: {e}")

    # Plot best Darts model
    if darts_best_pred is not None:
        fig = go.Figure()
        ctx = ts_tv_w.iloc[-56:]
        fig.add_trace(go.Scatter(x=ctx["ds"],y=ctx["y"],name="Context",line=dict(color="#BBDEFB",width=1.5)))
        fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=2.5)))
        fig.add_trace(go.Scatter(x=ts_test["ds"],y=darts_best_pred[:len(ts_test)],
            name="Temperature Forecasting Darts",line=dict(color="#F44336",width=2.5,dash="dot")))
        fig.update_layout(title="Darts 28-day Forecast vs Actual",
            template="plotly_white",legend=dict(orientation="h",y=-0.25))
        fig.show()
except Exception as e:
    print(f"Darts section failed: {e}")
    darts_best_pred = sn

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("\nTOP 3:")
print(top3.to_string(index=False))
fig = px.bar(results_df,x="model",y="RMSE",title="Model Comparison — RMSE",
    color="RMSE",color_continuous_scale="RdYlGn_r",text=results_df["RMSE"].round(3))
fig.update_layout(xaxis_tickangle=-35,template="plotly_white")
fig.show()

## Final Evaluation of Top 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=3)))
colors3 = ["#F44336","#4CAF50","#FF9800"]
for i,(_, row) in enumerate(top3.iterrows()):
    mname = row["model"]
    if "FLAML" in mname: pred_vals = flaml_pred
    elif "Darts" in mname: pred_vals = darts_best_pred if "darts_best_pred" in dir() else sn
    elif "Seasonal" in mname: pred_vals = sn
    elif "Moving" in mname: pred_vals = ma
    else: pred_vals = naive_p
    if pred_vals is not None:
        fig.add_trace(go.Scatter(x=ts_test["ds"],y=pred_vals[:TEST_SIZE],
            name=f"#{i+1} {mname}  RMSE={row['RMSE']:.3f}",
            line=dict(width=2.5,dash="dot",color=colors3[i])))
fig.update_layout(title="Top 3 Models — Forecast vs Actual",template="plotly_white",
    legend=dict(orientation="h",y=-0.25))
fig.show()

## Error Analysis

In [ ]:
best_name = top3.iloc[0]["model"]
if "FLAML" in best_name: best_pred_arr = flaml_pred
elif "Darts" in best_name: best_pred_arr = darts_best_pred
elif "sn_vals" in dir(): best_pred_arr = sn_vals
else: best_pred_arr = sn
actual_v = ts_test["y"].values
nc = min(len(actual_v),len(best_pred_arr))
errors = actual_v[:nc] - best_pred_arr[:nc]
print(f"Best: {best_name}  |  mean_err={errors.mean():.3f}  std={errors.std():.3f}")
fig, axes = plt.subplots(1,3,figsize=(18,4))
axes[0].hist(errors,bins=20,edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].axvline(0,color="red",ls="--"); axes[0].set_title("Error Distribution")
axes[1].plot(range(nc),errors,"o-",ms=4,color="#F44336")
axes[1].axhline(0,color="black",ls="--",lw=1); axes[1].set_title("Errors Over Horizon")
axes[2].scatter(actual_v[:nc],best_pred_arr[:nc],s=25,alpha=0.7,color="#388E3C")
mn=min(actual_v[:nc].min(),best_pred_arr[:nc].min())
mx=max(actual_v[:nc].max(),best_pred_arr[:nc].max())
axes[2].plot([mn,mx],[mn,mx],"r--")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted"); axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()

## Interpretation & Insights
1. **Annual cycle dominates** — Delhi swings from ~8°C (January) to ~43°C (May)
2. **Monsoon onset** (June–July) arrests the temperature rise
3. **Darts ExponentialSmoothing with seasonality** captures the annual wave
4. **Low noise** — Delhi temperature is unusually predictable against seasonal baseline
5. **lag_365 feature** is the single most predictive tabular feature

## Limitations
1. Only 4 years of data — annual lag features have limited training rows
2. No climate change trend component modelled
3. Station-level single-city data — not generalizable without retraining
4. Monsoon onset timing varies year-to-year by ±2 weeks
5. No humidity or wind speed as co-predictors

## How to Improve
1. Add lag_365 explicitly and measure RMSE drop
2. Add humidity as an exogenous regressor
3. Extend to 90-day horizon for agricultural planning
4. Use MSTL with two periods: 7 (weekly residual) and 365 (annual)
5. Automate monsoon onset date detection for annual adjustment

## Production Considerations
1. Retrain daily with latest NOAA/IMD readings
2. Publish 14-day forecast for irrigation advisory portal
3. Alert when forecast > 40°C for occupational health advisories
4. Validate against IMD 5-day official forecast

## Common Mistakes
1. Using season_length=7 for a climate series — weekly seasonality is negligible vs annual
2. Forgetting to load test CSV — some versions split train/test into separate files
3. Including humidity in the target instead of as a feature
4. MAPE on near-zero winter temperatures

## Mini Challenges
1. **Humidity model** — add `humidity` as covariate; does RMSE drop?
2. **Annual lag** — add lag_365; compare RMSE with and without it
3. **90-day horizon** — extend to 3 months; how does accuracy degrade?
4. **City comparison** — get another city's temperature dataset and compare patterns
5. **Degree-day energy forecast** — convert temperature to cooling/heating degree days

## Final Summary

### What We Built
- Loaded temperature forecasting dataset and built a time series
- Validated quality, Winsorised extremes, ran full EDA
- Benchmarked Naive, Seasonal Naive, MA, FLAML, and Darts
- Selected top 3 by RMSE with error analysis

### Key Takeaways
- Domain-specific seasonality patterns revealed by decomposition
- Darts captures trend and seasonality more effectively than flat baselines
- FLAML provides a competitive tabular ML benchmark
- Error analysis reveals systematic biases for targeted improvement

---